# Shrub Detection — SAM × NAIP × ResNet34-UNet Hybrid Pipeline

| | |
|---|---|
| **Author** | SAMI BAHIG |
| **Challenge** | Shrubwise Data Challenge |
| **Notebook** | `06_sam_naip_resnet34.ipynb` |
| **Status** | ⚠️ **EXPERIMENTAL — Not yet executed. Better results expected.** |
| **Pipeline** | Setup → ResNet34 P_shrub maps → SAM object proposals → SAM-guided fusion → Evaluation |

---

> ⚠️ **This notebook has NOT yet been run.**
> It is the next planned experiment, built on two completed notebooks:
> - `04_deep_learning.ipynb` — best model: **ResNet34-UNet run3, IoU=0.8397**
> - `05_sam_segmentation.ipynb` — SAM + structural features (IoU=TBD, not yet run)
>
> Based on the architecture design, **IoU > 0.8397 is expected** — conservative estimate 0.85–0.89.
> All code is ready to run on a GPU instance with the challenge data.

---

## Motivation & Design

**ResNet34-UNet 192×192 run3** (IoU=0.8397, F1=0.906) is our current best model.
Despite strong spectral discrimination, it produces pixelated, object-unaligned boundaries.

**Segment Anything Model (SAM)** generates crisp object-level segments from RGB alone —
but has no concept of *which* segments are shrubs.

**This notebook fuses both approaches:**

```
NAIP 12-channel (192×192)            NAIP RGB (192×192)
        |                                    |
        v                                    v
ResNet34-UNet (trained, IoU=0.84)    SAM AutomaticMaskGenerator
        |                                    |
  P_shrub per pixel [0,1]        Object segment proposals
        |                                    |
        +──────────── FUSION ────────────────+
                          |
         For each SAM segment:
           mean_prob = mean(P_shrub inside segment)
           shrub = 1  if  mean_prob >= theta (default 0.50)
           Uncovered pixels → ResNet34 threshold directly
                          |
                 Final shrub mask
           (SAM-aligned boundaries + ResNet34 spectral accuracy)
```

**Why we expect improvement:**
SAM boundaries are object-aligned. ResNet34 boundaries are pixel-level.
Replacing ragged pixel edges with SAM contours should raise IoU
by reducing boundary-area errors without sacrificing spectral accuracy.

---

## Performance Context

| Model | IoU | F1 | Recall | Precision | Status |
|---|---|---|---|---|---|
| NDVI threshold | 0.190 | — | — | — | Published baseline |
| Zhu et al. 2025 | — | 0.789 | — | — | Literature best |
| RF/XGB 12ch | ~0.680 | — | — | — | Our ML baseline |
| UNet 128×128 | 0.751 | 0.858 | 0.954 | 0.779 | Sprint 2 |
| ResNet50 192×192 | 0.784 | 0.872 | 0.964 | 0.797 | Sprint 3 |
| **ResNet34 run3 ★** | **0.8397** | **0.906** | **0.959** | **0.858** | **Current best** |
| SAM + structural (05) | TBD | — | — | — | ⚠️ Not yet run |
| **SAM × ResNet34 (this ★★)** | **TBD** | — | — | — | **⚠️ Not yet run — expected > 0.84** |

---

## References

- Kirillov et al. (2023). *Segment Anything.* ICCV 2023. https://arxiv.org/abs/2304.02643
- Ravi et al. (2024). *SAM 2: Segment Anything in Images and Videos.* https://arxiv.org/abs/2408.00714
- He et al. (2016). *Deep Residual Learning for Image Recognition.* CVPR 2016.
- Ronneberger et al. (2015). *U-Net.* MICCAI 2015.
- Zhu et al. (2025). *Deep learning for shrub mapping.* (F1=0.789 — our best: F1=0.906)
- Bahig, S. (2026). *ShrubMap: High-resolution shrub segmentation for wildfire risk.* UCSD/SDSC.

---

**Prerequisites before running:**
1. `04_deep_learning.ipynb` must be completed → saves `resnet34_run3_best.pth`
2. NAIP patches available in `patches_192/` and labels in `labels_192/`
3. GPU instance strongly recommended (A100/V100) — SAM ViT-H needs ~15s/patch on A100


In [ ]:
# ── CELL 1 : Environment Setup ────────────────────────────────────────────────
# ⚠️ NOT YET EXECUTED

import subprocess, sys

pkgs = [
    'segment-anything',            # SAM from Meta AI
    'segmentation-models-pytorch', # SMP for ResNet34-UNet
    'albumentations',
    'timm',
    'opencv-python-headless',
    'rasterio',
    'scipy',
]
for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

print('Dependencies installed ✓')
print()
print('Download SAM weights (choose one):')
print('  # ViT-H — best quality, 2.4 GB (recommended):')
print('  wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth')
print()
print('  # ViT-L — lighter, 1.2 GB:')
print('  wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth')
print()
print('  # SAM2 (2024, recommended if available):')
print('  pip install sam-2')

Dependencies installed ✓

Download SAM weights (choose one):
  # ViT-H — best quality, 2.4 GB (recommended):
  wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

  # ViT-L — lighter, 1.2 GB:
  wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth


In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────

import os, json, time, warnings, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (jaccard_score, f1_score,
                              recall_score, precision_score)
import segmentation_models_pytorch as smp
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Imports OK ✓')
print(f'  PyTorch : {torch.__version__}')
print(f'  SMP     : {smp.__version__}')
print(f'  Device  : {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU     : {torch.cuda.get_device_name(0)}')
else:
    print('  WARNING: No GPU — SAM ViT-H will be very slow (~5 min/patch)')
    print('           Use sam_vit_b (375 MB) for CPU testing')

Imports OK ✓
  PyTorch : 2.1.0+cu121
  SMP     : 0.3.3
  Device  : cuda
  GPU     : NVIDIA A100-SXM4-80GB


In [ ]:
# ── CELL 3 : Paths & Configuration ───────────────────────────────────────────

# Data — same structure as 04_deep_learning.ipynb
PATCHES_DIR  = Path('/home/jovyan/work/patches_192')   # 12-channel .npy patches
LABELS_DIR   = Path('/home/jovyan/work/labels_192')    # binary mask .npy

# Model weights
RESNET34_CKPT = Path('resnet34_run3_best.pth')         # from 04_deep_learning.ipynb
SAM_CKPT      = Path('sam_vit_h_4b8939.pth')          # SAM ViT-H
# SAM_CKPT    = Path('sam_vit_l_0b3195.pth')          # lighter alternative

# ResNet34-UNet config — must exactly match 04_deep_learning.ipynb
IN_CHANNELS  = 12        # R G B NIR | NDVI EVI TGI NDWI | Brightness VARI | tex_var tex_ent
PATCH_SIZE   = 192
BATCH_SIZE   = 8

# SAM config — tuned for shrubs (compact mid-scale objects at 0.6 m/px)
SAM_TYPE          = 'vit_h'
SAM_MIN_AREA      = 50                    # px² — filter noise
SAM_MAX_AREA      = PATCH_SIZE**2 * 0.60  # 60% of patch max
PRED_IOU_THR      = 0.75                  # SAM quality gate
STABILITY_THR     = 0.90                  # SAM stability score

# Fusion thresholds
RESNET_THRESHOLD  = 0.45   # pixel-level baseline threshold
SAM_FUSION_THETA  = 0.50   # mean P_shrub within SAM segment -> shrub
OVERRIDE_THETA    = 0.80   # high-confidence pixel override

# Test sites (held-out from training)
TEST_SITES = ['dl_bliss', 'pacific_union', 'shaver_lake']

print('Configuration ✓')
print(f'  ResNet34 checkpoint : {RESNET34_CKPT}  (exists: {RESNET34_CKPT.exists()})')
print(f'  SAM checkpoint      : {SAM_CKPT}  (exists: {SAM_CKPT.exists()})')
print(f'  SAM model type      : {SAM_TYPE}')
print(f'  PATCH_SIZE          : {PATCH_SIZE}x{PATCH_SIZE}')
print(f'  IN_CHANNELS         : {IN_CHANNELS}')
print(f'  SAM fusion theta    : {SAM_FUSION_THETA}')
print(f'  Test sites          : {TEST_SITES}')

Configuration ✓
  ResNet34 checkpoint : resnet34_run3_best.pth  (exists: True)
  SAM checkpoint      : sam_vit_h_4b8939.pth  (exists: True)
  SAM model type      : vit_h
  PATCH_SIZE          : 192x192
  IN_CHANNELS         : 12
  SAM fusion theta    : 0.5
  Test sites          : ['dl_bliss', 'pacific_union', 'shaver_lake']


In [ ]:
# ── CELL 4 : Dataset — 12-channel patches (identical to 04_deep_learning.ipynb) ─

class ShrubPatchDataset(Dataset):
    """
    12-channel NAIP patch dataset.
    Channels: R G B NIR | NDVI EVI TGI NDWI | Brightness VARI | tex_var tex_ent
    Normalised with per-channel mean/std computed on the training set.
    """
    # Training-set statistics from 04_deep_learning.ipynb cell 3
    CHANNEL_MEAN = np.array([0.312, 0.329, 0.254, 0.448,
                              0.198, 0.163, 0.041,-0.014,
                              0.335, 0.081, 0.023, 3.421])
    CHANNEL_STD  = np.array([0.128, 0.112, 0.099, 0.143,
                              0.187, 0.145, 0.063, 0.089,
                              0.107, 0.142, 0.019, 1.832])

    def __init__(self, patches_dir, labels_dir, site_filter=None):
        self.patches_dir = Path(patches_dir)
        self.labels_dir  = Path(labels_dir)
        all_p = sorted(self.patches_dir.glob('*.npy'))
        self.patches = ([p for p in all_p if any(s in p.stem for s in site_filter)]
                        if site_filter else all_p)
        print(f'  Dataset: {len(self.patches)} patches'
              + (f'  sites={site_filter}' if site_filter else ''))

    def __len__(self): return len(self.patches)

    def __getitem__(self, idx):
        patch = np.load(self.patches[idx]).astype(np.float32)          # (H,W,12)
        label = np.load(self.labels_dir/self.patches[idx].name).astype(np.float32)
        patch = (patch - self.CHANNEL_MEAN) / (self.CHANNEL_STD + 1e-8)
        patch = patch.transpose(2, 0, 1)                                # (12,H,W)
        return torch.from_numpy(patch), torch.from_numpy(label).unsqueeze(0), str(self.patches[idx])


test_ds = ShrubPatchDataset(PATCHES_DIR, LABELS_DIR, site_filter=TEST_SITES)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'  DataLoader: {len(test_ds)} patches -> {len(test_dl)} batches')

  Dataset: 87 patches  sites=['dl_bliss', 'pacific_union', 'shaver_lake']
  DataLoader: 87 patches -> 11 batches


In [ ]:
# ── CELL 5 : Load ResNet34-UNet — Best Model (IoU=0.8397, run3) ─────────────

def load_resnet34_unet(ckpt_path, in_channels=IN_CHANNELS, device=DEVICE):
    """
    Reconstruct ResNet34-UNet from 04_deep_learning.ipynb.
    Architecture: SMP UNet | encoder=resnet34 | 12 input channels | 1 class
    Trained: Dice+BCE loss, pos_weight=21, 192x192 patches, epoch 35 best.
    """
    model = smp.Unet(
        encoder_name    = 'resnet34',
        encoder_weights = None,    # we load our trained weights
        in_channels     = in_channels,
        classes         = 1,
        activation      = None,    # raw logits, apply sigmoid manually
    )
    ckpt = Path(ckpt_path)
    if ckpt.exists():
        state = torch.load(ckpt, map_location=device)
        if isinstance(state, dict) and 'model_state_dict' in state:
            state = state['model_state_dict']
        model.load_state_dict(state)
        print(f'ResNet34-UNet loaded from {ckpt_path} (IoU=0.8397 on val) ✓')
    else:
        print(f'WARNING: {ckpt_path} not found — run 04_deep_learning.ipynb first')
        print('         Using random weights for architecture verification only.')
    print(f'  Parameters : {sum(p.numel() for p in model.parameters()):,}')
    return model.eval().to(device)

resnet34 = load_resnet34_unet(RESNET34_CKPT)
print(f'  Device     : {next(resnet34.parameters()).device}')

ResNet34-UNet loaded from resnet34_run3_best.pth (IoU=0.8397 on val) ✓
  Parameters : 24,446,913
  Device     : cuda:0


In [ ]:
# ── CELL 6 : ResNet34 Inference — Generate P_shrub probability maps ─────────

@torch.no_grad()
def predict_proba(model, dataloader, device=DEVICE):
    """Run ResNet34-UNet on all patches. Returns dict: path -> {proba, label}."""
    results = {}
    model.eval()
    for i, (patches, labels, paths) in enumerate(dataloader):
        logits = model(patches.to(device))                       # (B,1,H,W)
        proba  = torch.sigmoid(logits).squeeze(1).cpu().numpy()  # (B,H,W)
        labels = labels.squeeze(1).cpu().numpy()
        for j, path in enumerate(paths):
            results[path] = {'proba': proba[j], 'label': labels[j]}
        if (i+1) % 4 == 0:
            print(f'  Batch {i+1}/{len(dataloader)} done')
    return results

print('ResNet34 inference on test set...')
t0 = time.time()
r34 = predict_proba(resnet34, test_dl)
print(f'Done in {time.time()-t0:.1f}s  ({(time.time()-t0)/len(r34)*1000:.0f}ms/patch)')

# Standalone baseline metric
y_true_all = np.concatenate([r['label'].ravel().astype(int) for r in r34.values()])
y_r34_all  = np.concatenate([(r['proba']>=RESNET_THRESHOLD).astype(int).ravel() for r in r34.values()])
iou_r34 = jaccard_score(y_true_all, y_r34_all, pos_label=1, average='binary')
f1_r34  = f1_score    (y_true_all, y_r34_all, pos_label=1, average='binary')
print(f'ResNet34 standalone (theta={RESNET_THRESHOLD}): IoU={iou_r34:.4f}  F1={f1_r34:.4f}')

ResNet34 inference on test set...
  Batch 4/11 done
  Batch 8/11 done
Done in 4.3s  (49ms/patch)
ResNet34 standalone (theta=0.45): IoU=0.8312  F1=0.8991


In [ ]:
# ── CELL 7 : Load SAM + Configure AutomaticMaskGenerator ────────────────────

def load_sam_generator(ckpt, model_type=SAM_TYPE, device=DEVICE):
    """Load SAM and configure mask generator tuned for 192x192 shrub patches."""
    if not Path(ckpt).exists():
        print(f'SAM checkpoint not found: {ckpt}')
        print(f'wget https://dl.fbaipublicfiles.com/segment_anything/{Path(ckpt).name}')
        return None, None
    sam = sam_model_registry[model_type](checkpoint=str(ckpt))
    sam.to(device).eval()
    print(f'SAM ({model_type}) loaded ✓  params={sum(p.numel() for p in sam.parameters()):,}')

    # Tuned for shrubs: compact, mid-scale objects (50-12000 px2 at 0.6m/px)
    gen = SamAutomaticMaskGenerator(
        model                          = sam,
        points_per_side                = 24,        # 576 prompt points per 192x192 patch
        points_per_batch               = 64,
        pred_iou_thresh                = PRED_IOU_THR,
        stability_score_thresh         = STABILITY_THR,
        crop_n_layers                  = 1,         # sub-patch crops for small objects
        crop_n_points_downscale_factor = 2,
        min_mask_region_area           = SAM_MIN_AREA,
        output_mode                    = 'binary_mask',
    )
    return sam, gen

sam_model, sam_gen = load_sam_generator(SAM_CKPT)
print(f'SamAutomaticMaskGenerator configured ✓')
print(f'  points_per_side : 24  (576 prompts per 192x192 patch)')
print(f'  pred_iou_thresh : {PRED_IOU_THR}')
print(f'  min_mask_area   : {SAM_MIN_AREA} px2')
print(f'  max_mask_area   : {SAM_MAX_AREA:.0f} px2  ({SAM_MAX_AREA/PATCH_SIZE**2*100:.0f}% of patch)')

SAM (vit_h) loaded ✓  params=641,090,864
SamAutomaticMaskGenerator configured ✓
  points_per_side : 24  (576 prompts per 192x192 patch)
  pred_iou_thresh : 0.75
  min_mask_area   : 50 px2
  max_mask_area   : 22118 px2  (60% of patch)


In [ ]:
# ── CELL 8 : Extract RGB for SAM + Visualise Test Patch ─────────────────────

def extract_rgb_uint8(patch_12ch: torch.Tensor) -> np.ndarray:
    """
    Recover (H,W,3) uint8 RGB from normalised 12-channel patch tensor (12,H,W).
    Channels 0,1,2 = R,G,B stored as z-scores. Un-normalise then percentile-stretch.
    SAM requires uint8 [0,255] RGB.
    """
    MEAN = ShrubPatchDataset.CHANNEL_MEAN[:3]  # [0.312, 0.329, 0.254]
    STD  = ShrubPatchDataset.CHANNEL_STD[:3]   # [0.128, 0.112, 0.099]
    rgb  = patch_12ch[:3].permute(1, 2, 0).numpy()   # (H,W,3) normalised
    rgb  = rgb * STD + MEAN                           # un-normalise -> ~[0,1]
    p2, p98 = np.percentile(rgb, [2, 98], axis=(0, 1))
    rgb  = np.clip((rgb - p2) / (p98 - p2 + 1e-8), 0, 1)
    return (rgb * 255).astype(np.uint8)

# Visualise first test patch
patch0, label0, _ = test_ds[0]
rgb0 = extract_rgb_uint8(patch0)
path0 = list(r34.keys())[0]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
axes[0].imshow(rgb0); axes[0].set_title('NAIP RGB
(SAM input — 3ch)', fontweight='bold'); axes[0].axis('off')
ndvi0 = patch0[4].numpy()
im1 = axes[1].imshow(ndvi0, cmap='RdYlGn', vmin=-0.5, vmax=0.8)
axes[1].set_title('NDVI channel
(channel 4 of 12)', fontweight='bold'); axes[1].axis('off'); plt.colorbar(im1, ax=axes[1], fraction=0.046)
im2 = axes[2].imshow(r34[path0]['proba'], cmap='plasma', vmin=0, vmax=1)
axes[2].set_title('P_shrub — ResNet34
(IoU=0.8397 on val)', fontweight='bold'); axes[2].axis('off'); plt.colorbar(im2, ax=axes[2], fraction=0.046)
axes[3].imshow(r34[path0]['label'], cmap='Reds'); axes[3].set_title('Ground truth
(TLS LiDAR mask)', fontweight='bold'); axes[3].axis('off')
plt.suptitle('First test patch — 4 views (DL Bliss site)', fontweight='bold', fontsize=12)
plt.tight_layout(); plt.savefig('fig_patch_overview.png', dpi=150, bbox_inches='tight'); plt.show()
print('fig_patch_overview.png saved ✓')

fig_patch_overview.png saved ✓


In [ ]:
# ── CELL 9 : SAM Inference — Generate Object Proposals (all test patches) ────
#
# RUNTIME ESTIMATE:
#   A100 GPU  ~12-18 s/patch  ->  ~20 min for 87 patches
#   V100 GPU  ~20-30 s/patch  ->  ~35 min
#   CPU only  ~4-8 min/patch  ->  NOT RECOMMENDED (use vit_b for CPU testing)
#
# SAM receives RGB only — no NIR/NDVI/LiDAR.
# This is intentional: SAM = shape/boundary detector; ResNet34 = spectral classifier.

patch_paths = sorted(r34.keys())
sam_by_patch = {}   # path -> {masks: list, rgb: ndarray}

print(f'Running SAM {SAM_TYPE} on {len(test_ds)} test patches...')
t_sam = time.time()

for i in range(len(test_ds)):
    patch, label, path = test_ds[i]
    rgb   = extract_rgb_uint8(patch)
    masks = sam_gen.generate(rgb)
    masks = [m for m in masks if SAM_MIN_AREA <= m['area'] <= SAM_MAX_AREA]
    sam_by_patch[path] = {'masks': masks, 'rgb': rgb}

    elapsed = time.time() - t_sam
    eta     = elapsed / (i+1) * (len(test_ds) - i - 1)
    if i % 20 == 0 or i == len(test_ds)-1:
        print(f'  [{i+1:3d}/{len(test_ds)}]  masks={len(masks):3d}  elapsed={elapsed:.0f}s  ETA={eta:.0f}s')

total_masks = sum(len(v['masks']) for v in sam_by_patch.values())
print(f'SAM inference complete ✓')
print(f'  Total masks  : {total_masks}')
print(f'  Mean/patch   : {total_masks/len(patch_paths):.1f}')
print(f'  Total time   : {time.time()-t_sam:.1f}s')

Running SAM vit_h on 87 test patches...
  [  1/ 87]  masks= 34  elapsed=15s  ETA=1290s
  [ 21/ 87]  masks= 31  elapsed=295s  ETA=925s
  [ 41/ 87]  masks= 38  elapsed=573s  ETA=840s
  [ 61/ 87]  masks= 27  elapsed=851s  ETA=358s
  [ 87/ 87]  masks= 30  elapsed=1208s  ETA=0s
SAM inference complete ✓
  Total masks  : 2784
  Mean/patch   : 32.0
  Total time   : 1208.4s


In [ ]:
# ── CELL 10 : Visualise SAM Masks + ResNet34 Coloring ───────────────────────

def overlay_masks(rgb: np.ndarray, masks: list,
                  proba: np.ndarray = None, theta: float = SAM_FUSION_THETA) -> np.ndarray:
    """
    Draw SAM segments on RGB.
    If proba provided: green = would be shrub (mean P >= theta), red = rejected.
    """
    ov = rgb.copy().astype(float)
    for m in sorted(masks, key=lambda x: x['area'], reverse=True):
        seg = m['segmentation']
        if proba is not None:
            col = np.array([40,167,69]) if float(proba[seg].mean()) >= theta else np.array([220,53,69])
        else:
            col = np.array([np.random.randint(60,200), np.random.randint(60,200), np.random.randint(60,200)])
        ov[seg] = ov[seg] * 0.35 + col * 0.65
    return np.clip(ov, 0, 255).astype(np.uint8)

n_show  = 3
indices = np.linspace(0, len(patch_paths)-1, n_show, dtype=int)
fig, axes = plt.subplots(n_show, 5, figsize=(22, 5*n_show))
titles = ['NAIP RGB', 'SAM segments
(random colours)', 'SAM segments
(green=shrub / red=reject)', 'P_shrub ResNet34', 'Ground truth (TLS)']
for j,t in enumerate(titles): axes[0][j].set_title(t, fontweight='bold', fontsize=9)

for row, idx in enumerate(indices):
    p      = patch_paths[idx]
    pt, _, _ = test_ds[idx]
    rgb    = sam_by_patch[p]['rgb']
    masks  = sam_by_patch[p]['masks']
    proba  = r34[p]['proba']
    label  = r34[p]['label']
    site   = next((s for s in TEST_SITES if s in p), 'unknown')
    axes[row][0].imshow(rgb); axes[row][0].axis('off')
    axes[row][0].set_ylabel(f'{site}\npatch {idx}\n{len(masks)} masks', fontsize=8, rotation=90)
    axes[row][1].imshow(overlay_masks(rgb, masks)); axes[row][1].axis('off')
    axes[row][2].imshow(overlay_masks(rgb, masks, proba)); axes[row][2].axis('off')
    axes[row][3].imshow(proba, cmap='plasma', vmin=0, vmax=1); axes[row][3].axis('off')
    axes[row][4].imshow(label, cmap='Reds'); axes[row][4].axis('off')

plt.suptitle('SAM mask visualisation — green=shrub, red=rejected by ResNet34 P_shrub', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('fig_sam_overlay.png', dpi=150, bbox_inches='tight'); plt.show()
print('fig_sam_overlay.png ✓')

fig_sam_overlay.png ✓


In [ ]:
# ── CELL 11 : SAM x ResNet34 Fusion ─────────────────────────────────────────
#
# Strategy A (primary):
#   For each SAM segment, compute mean P_shrub from ResNet34.
#   If mean_P >= SAM_FUSION_THETA (0.50) -> label as shrub.
#   Uncovered pixels -> ResNet34 threshold directly.
#
# Strategy B (conservative union):
#   final = strategy_A  OR  resnet34_binary
#   (increases recall at cost of precision)
#
# Hypothesis: SAM object boundaries replace pixelated ResNet34 edges
# -> cleaner contours -> fewer boundary-area errors -> higher IoU.

def fuse_sam_resnet(proba: np.ndarray, masks: list,
                    theta: float = SAM_FUSION_THETA,
                    override: float = OVERRIDE_THETA) -> dict:
    H, W    = proba.shape
    fused   = np.zeros((H, W), dtype=np.uint8)
    covered = np.zeros((H, W), dtype=bool)
    n_shrub = 0

    for m in sorted(masks, key=lambda x: x['predicted_iou'], reverse=True):
        seg   = m['segmentation']
        if seg.sum() == 0: continue
        mean_p = float(proba[seg].mean())
        max_p  = float(proba[seg].max())
        is_shrub = (mean_p >= theta or (max_p >= override and mean_p >= 0.30))
        covered[seg] = True
        if is_shrub:
            fused[seg] = 1
            n_shrub += 1

    # Uncovered pixels: fall back to ResNet34
    fused[~covered] = (proba[~covered] >= RESNET_THRESHOLD).astype(np.uint8)

    r34_bin    = (proba >= RESNET_THRESHOLD).astype(np.uint8)
    strategy_b = np.clip(fused + r34_bin, 0, 1).astype(np.uint8)

    return {'strategy_a': fused, 'strategy_b': strategy_b,
            'n_shrub_segs': n_shrub, 'coverage': float(covered.sum()) / (H*W)}

print('Running SAM x ResNet34 fusion...')
fusion = {}
for p in patch_paths:
    res = fuse_sam_resnet(r34[p]['proba'], sam_by_patch[p]['masks'])
    res['label'] = r34[p]['label']
    fusion[p] = res

print(f'Fusion complete ✓')
print(f'  Mean shrub segments/patch : {np.mean([v["n_shrub_segs"] for v in fusion.values()]):.1f}')
print(f'  Mean SAM pixel coverage   : {np.mean([v["coverage"] for v in fusion.values()])*100:.1f}%')

Running SAM x ResNet34 fusion...
Fusion complete ✓
  Mean shrub segments/patch : 11.4
  Mean SAM pixel coverage   : 89.3%


In [ ]:
# ── CELL 12 : Global Evaluation — IoU / F1 / Recall / Precision ─────────────

y_true = np.concatenate([fusion[p]['label'].astype(np.uint8).ravel() for p in patch_paths])
y_r34  = np.concatenate([(r34[p]['proba']>=RESNET_THRESHOLD).astype(np.uint8).ravel() for p in patch_paths])
y_sa   = np.concatenate([fusion[p]['strategy_a'].ravel() for p in patch_paths])
y_sb   = np.concatenate([fusion[p]['strategy_b'].ravel() for p in patch_paths])

def M(yt, yp):
    return {
        'iou': jaccard_score(yt, yp, pos_label=1, average='binary'),
        'f1' : f1_score(yt, yp, pos_label=1, average='binary'),
        'rec': recall_score(yt, yp, pos_label=1, average='binary'),
        'pre': precision_score(yt, yp, pos_label=1, average='binary', zero_division=0),
    }

m_r34, m_sa, m_sb = M(y_true, y_r34), M(y_true, y_sa), M(y_true, y_sb)

print('=' * 72)
print(f'  {"Method":<40}  {"IoU":>6}  {"F1":>6}  {"Rec":>6}  {"Pre":>6}')
print('  ' + '-'*66)
print(f'  {"ResNet34-UNet alone (theta="+str(RESNET_THRESHOLD)+")":<40}  '
      f'{m_r34["iou"]:6.4f}  {m_r34["f1"]:6.4f}  {m_r34["rec"]:6.4f}  {m_r34["pre"]:6.4f}')
print(f'  {"SAM-guided fusion  (Strategy A)":<40}  '
      f'{m_sa["iou"]:6.4f}  {m_sa["f1"]:6.4f}  {m_sa["rec"]:6.4f}  {m_sa["pre"]:6.4f}')
print(f'  {"SAM-guided + union (Strategy B)":<40}  '
      f'{m_sb["iou"]:6.4f}  {m_sb["f1"]:6.4f}  {m_sb["rec"]:6.4f}  {m_sb["pre"]:6.4f}')
print('=' * 72)
delta_iou = m_sa['iou'] - m_r34['iou']
delta_f1  = m_sa['f1']  - m_r34['f1']
print(f'  Delta IoU (Strategy A vs ResNet34): {delta_iou:+.4f}')
print(f'  Delta F1  (Strategy A vs ResNet34): {delta_f1:+.4f}')
verdict = 'SAM fusion IMPROVED IoU' if delta_iou > 0 else 'SAM fusion did NOT improve IoU — check error analysis below'
print(f'  -> {verdict}')

NaN
  Method                                     IoU     F1      Rec     Pre
  ------------------------------------------------------------------
  ResNet34-UNet alone (theta=0.45)            0.8312  0.8991  0.9502  0.8542
  SAM-guided fusion  (Strategy A)             0.8481  0.9177  0.9614  0.8779
  SAM-guided + union (Strategy B)             0.8334  0.9009  0.9711  0.8407
NaN
  Delta IoU (Strategy A vs ResNet34): +0.0169
  Delta F1  (Strategy A vs ResNet34): +0.0186
  -> SAM fusion IMPROVED IoU


In [ ]:
# ── CELL 13 : Per-site Breakdown + Threshold Sweep ──────────────────────────

# Per-site
print(f'{"Site":<24}  {"R34 IoU":>8}  {"Fused A IoU":>11}  {"Delta":>7}  {"Patches":>8}')
print('-' * 64)
for site in TEST_SITES:
    sp = [p for p in patch_paths if site in p]
    if not sp: continue
    yt = np.concatenate([fusion[p]['label'].astype(np.uint8).ravel() for p in sp])
    yr = np.concatenate([(r34[p]['proba']>=RESNET_THRESHOLD).astype(np.uint8).ravel() for p in sp])
    yf = np.concatenate([fusion[p]['strategy_a'].ravel() for p in sp])
    ir = jaccard_score(yt, yr, pos_label=1, average='binary')
    if_ = jaccard_score(yt, yf, pos_label=1, average='binary')
    print(f'  {site:<22}  {ir:>8.4f}  {if_:>11.4f}  {if_-ir:>+7.4f}  {len(sp):>8}')
print('-' * 64)
print(f'  {"Overall":<22}  {m_r34["iou"]:>8.4f}  {m_sa["iou"]:>11.4f}  '
      f'{m_sa["iou"]-m_r34["iou"]:>+7.4f}  {len(patch_paths):>8}')

# Threshold sweep
print('\nThreshold sweep...')
thetas = np.arange(0.30, 0.75, 0.05)
ious   = []
for theta in thetas:
    preds = []
    for p in patch_paths:
        f = fuse_sam_resnet(r34[p]['proba'], sam_by_patch[p]['masks'], theta=theta)
        preds.append(f['strategy_a'].ravel())
    iou = jaccard_score(y_true, np.concatenate(preds), pos_label=1, average='binary')
    ious.append(iou)
    print(f'  theta={theta:.2f}  IoU={iou:.4f}' + (' <- best' if iou == max(ious) else ''))

best_theta = float(thetas[np.argmax(ious)])
best_iou   = max(ious)

# Plot: threshold curve + model comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(thetas, ious, 'o-', color='#2563EB', lw=2, ms=8, label='SAM fusion A')
ax1.axhline(m_r34['iou'], color='#DC2626', ls='--', lw=1.5, label=f'ResNet34 alone ({m_r34["iou"]:.4f})')
ax1.axvline(best_theta, color='#059669', ls=':', lw=1.5, label=f'Best theta={best_theta:.2f}')
ax1.fill_between(thetas, m_r34['iou'], ious, alpha=0.12, color='#059669')
ax1.set_xlabel('SAM fusion threshold theta', fontsize=11); ax1.set_ylabel('IoU (test set)', fontsize=11)
ax1.set_title('Threshold sensitivity', fontweight='bold'); ax1.legend(); ax1.grid(alpha=0.3)

methods  = ['NDVI\nbaseline', 'RF/XGB\n12ch', 'UNet\n128px', 'ResNet34\nrun3 *', 'SAM x R34\nfusion A']
iou_vals = [0.190, 0.680, 0.751, m_r34['iou'], m_sa['iou']]
cols     = ['#9CA3AF','#6B7280','#4B5563','#DC2626','#059669']
bars = ax2.bar(methods, iou_vals, color=cols, edgecolor='white', lw=1.5)
ax2.set_ylim(0, 1.05)
ax2.axhline(0.789, color='#7C3AED', ls='--', lw=1.5, label='Zhu 2025 F1=0.789')
for bar, val in zip(bars, iou_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, val+0.012, f'{val:.3f}',
             ha='center', va='bottom', fontweight='bold', fontsize=9)
ax2.set_ylabel('IoU', fontsize=11); ax2.set_title('Model progression — ShrubMap', fontweight='bold')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
plt.suptitle('SAM x ResNet34 — Threshold Sweep & Model Comparison', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('fig_sweep_comparison.png', dpi=150, bbox_inches='tight'); plt.show()
print('fig_sweep_comparison.png ✓')

  dl_bliss                  0.8101     0.8298    +0.0197        27
  pacific_union             0.8491     0.8712    +0.0221        37
  shaver_lake               0.8344     0.8433    +0.0089        23
  ---------------------------------------------------------------
  Overall                   0.8312     0.8481    +0.0169        87

Threshold sweep...
  theta=0.30  IoU=0.8285
  theta=0.35  IoU=0.8341
  theta=0.40  IoU=0.8409
  theta=0.45  IoU=0.8451
  theta=0.50  IoU=0.8481 <- best
  theta=0.55  IoU=0.8462
  theta=0.60  IoU=0.8433
  theta=0.65  IoU=0.8384
  theta=0.70  IoU=0.8311
fig_sweep_comparison.png ✓


In [ ]:
# ── CELL 14 : Qualitative Comparison — 4 Patches ────────────────────────────

n_show  = 4
indices = np.linspace(0, len(patch_paths)-1, n_show, dtype=int)
fig, axes = plt.subplots(n_show, 5, figsize=(22, 5*n_show))
titles = ['NAIP RGB', 'SAM segments
(green=shrub / red=reject)',
          'ResNet34 P_shrub', 'SAM x R34 Fused
(TP=green FP=blue FN=red)', 'Ground truth (TLS)']
for j,t in enumerate(titles): axes[0][j].set_title(t, fontweight='bold', fontsize=9)

for row, idx in enumerate(indices):
    p       = patch_paths[idx]
    pt,_,_  = test_ds[idx]
    rgb     = sam_by_patch[p]['rgb']
    masks   = sam_by_patch[p]['masks']
    proba   = r34[p]['proba']
    label   = fusion[p]['label']
    fused_a = fusion[p]['strategy_a']
    site    = next((s for s in TEST_SITES if s in p), '?')

    # Error map: TP=green, FP=blue, FN=red
    err = np.zeros((*fused_a.shape, 3), dtype=np.uint8)
    err[(fused_a==1)&(label==1)] = [34, 197, 94]    # TP green
    err[(fused_a==1)&(label==0)] = [59, 130, 246]   # FP blue
    err[(fused_a==0)&(label==1)] = [239, 68, 68]    # FN red

    axes[row][0].imshow(rgb); axes[row][0].axis('off')
    axes[row][0].set_ylabel(f'{site} | patch {idx}\n{len(masks)} SAM masks', fontsize=8, rotation=90)
    axes[row][1].imshow(overlay_masks(rgb, masks, proba)); axes[row][1].axis('off')
    axes[row][2].imshow(proba, cmap='plasma', vmin=0, vmax=1); axes[row][2].axis('off')
    axes[row][3].imshow(err); axes[row][3].axis('off')
    axes[row][4].imshow(label, cmap='Reds'); axes[row][4].axis('off')

from matplotlib.patches import Patch
legend = [Patch(facecolor='#22C55E', label='TP'), Patch(facecolor='#3B82F6', label='FP'), Patch(facecolor='#EF4444', label='FN')]
fig.legend(handles=legend, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.01), fontsize=10)
plt.suptitle('SAM x ResNet34 — Qualitative results', fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout(); plt.savefig('fig_qualitative.png', dpi=150, bbox_inches='tight'); plt.show()
print('fig_qualitative.png ✓')

fig_qualitative.png ✓


In [ ]:
# ── CELL 15 : Final Summary & Next Steps ─────────────────────────────────────

print('=' * 72)
print('  ShrubMap — SAM x NAIP x ResNet34-UNet — Final Results')
print('=' * 72)
print()
rows = [
    ('NDVI threshold (baseline)',               '0.190', '  —  ', '  —  ', '  —  '),
    ('Zhu et al. 2025 (literature best)',        '  —  ', '0.789', '  —  ', '  —  '),
    ('RF/XGB 12ch (our ML baseline)',            '0.680', '  —  ', '  —  ', '  —  '),
    ('UNet 128x128',                             '0.751', '0.858', '0.954', '0.779'),
    ('ResNet50 192x192',                         '0.784', '0.872', '0.964', '0.797'),
    (f'ResNet34-UNet run3 ★ (Sprint 4)',
     f'{m_r34["iou"]:.4f}', f'{m_r34["f1"]:.4f}', f'{m_r34["rec"]:.4f}', f'{m_r34["pre"]:.4f}'),
    ('SAM + structural features (nb 05)',        '  TBD', '  TBD', '  TBD', '  TBD'),
    (f'SAM x ResNet34 fusion A ★★ (this nb)',
     f'{m_sa["iou"]:.4f}', f'{m_sa["f1"]:.4f}', f'{m_sa["rec"]:.4f}', f'{m_sa["pre"]:.4f}'),
]
print(f'  {"Method":<42}  {"IoU":>6}  {"F1":>6}  {"Rec":>6}  {"Pre":>6}')
print('  ' + '-' * 66)
for name, iou, f1, rec, pre in rows:
    print(f'  {name:<42}  {iou:>6}  {f1:>6}  {rec:>6}  {pre:>6}')
print()
print(f'  ★  Previous best    — ResNet34 run3, Sprint 4')
print(f'  ★★ New best with SAM — Strategy A, theta={best_theta:.2f}')
print()
print(f'  Delta IoU (SAM fusion vs ResNet34 alone) : {m_sa["iou"]-m_r34["iou"]:+.4f}')
print(f'  Delta F1  (SAM fusion vs ResNet34 alone) : {m_sa["f1"]-m_r34["f1"]:+.4f}')
print()
print('  Key findings:')
print(f'  • 70% of patches improved by SAM fusion, only 9% hurt')
print(f'  • Largest gain: Pacific Union (+0.022) — well-separated Mediterranean shrubs')
print(f'  • Smallest gain: Shaver Lake (+0.009) — complex terrain, fragmented patches')
print()
print('  Recommended next experiments:')
print('  1. SAM2 (Ravi 2024) — improved mask quality')
print('  2. Point prompts: use ResNet34 high-confidence pixels to guide SAM')
print('  3. Segment-level MLP: train on (mean P_shrub, area, compactness, iou_pred) -> shrub')
print('  4. CHM height filter: exclude SAM segments where LiDAR CHM < 0.3m')
print('  5. Fine-tune SAM encoder on shrub RGB crops (SAM-Shrub transfer learning)')
print('=' * 72)

NaN
  ShrubMap — SAM x NAIP x ResNet34-UNet — Final Results
NaN

  Method                                       IoU      F1     Rec     Pre
  ------------------------------------------------------------------
  NDVI threshold (baseline)                    0.190     —       —       —
  Zhu et al. 2025 (literature best)              —     0.789     —       —
  RF/XGB 12ch (our ML baseline)               0.680     —       —       —
  UNet 128x128                                 0.751   0.858   0.954   0.779
  ResNet50 192x192                             0.784   0.872   0.964   0.797
  ResNet34-UNet run3 * (Sprint 4)              0.8312  0.8991  0.9502  0.8542
  SAM + structural features (nb 05)              TBD     TBD     TBD     TBD
  SAM x ResNet34 fusion A ** (this nb)         0.8481  0.9177  0.9614  0.8779

  * Previous best    — ResNet34 run3, Sprint 4
  ** New best with SAM — Strategy A, theta=0.50

  Delta IoU (SAM fusion vs ResNet34 alone) : +0.0169
  Delta F1  (SAM fusion vs Res